# B-09: recursive 365-day daily rollout backtest (does autoregressive forecasting compound error?)

Tests whether **recursive** (feed-your-own-prediction-back-in) forecasting stays usable over a full
365-day horizon, or compounds/degrades badly -- directly motivated by a sibling project's own
lesson (`FORECASTING_LEARNINGS.md`: "always validate a rollout mechanism against a real held-out
window before trusting it on a genuinely blind future window") and a direct precursor to the
long-range scenario work (D-46 requirement 5, D-52).

**Design**: a single fixed anchor date (**2021-12-16**, Tower 4 -- best real-data density in the
following year, 341/365 days pass the `y_observed`>=6h/day rule, though true hourly-observed
density in that window is only 48.9% -- state this caveat alongside every headline number, not as
a footnote), one continuous 365-day recursive chain per model -- **not** a walk-forward evaluation
across many re-originating origins (which is what `B03a_arima.ipynb`'s SARIMAX harness does, and
which never lets error compound past a few days since it re-observes real ground truth every
`stride`). Every model is fit fresh on data strictly <= the anchor (no reuse of B03/B03a/B03b's
existing <=2021-12-31 results -- that would leak real 2021-12-17...12-31 observations into
"training"). Exogenous/`fx_` drivers are fed as real historical values throughout (perfect
foresight) -- this isolates the recursive-mechanism question from driver-forecast realism (B-08's
separate, still-queued scope, D-47).

**Models (committed scope this session):** SARIMAX, RandomForest, XGBoost, LightGBM, DLinear, LSTM
-- Tower 4 only. TFT, Tower 9, and TabPFN-prep are stretch items (see end of notebook).

**Evaluation:** primary target `y_observed` only (never `y_gapfilled`, avoids circularity).
Non-overlapping lead-time bins (days 1-7, 8-30, 31-90, 91-180, 181-270, 271-365) rather than one
blended number -- the direct M5-lesson analogue of "don't blend across the hierarchy" (here,
lead-time-within-the-chain instead of store/category). Two baselines: chain-persistence (repeat
the anchor day's real value) and day-of-year-windowed climatology (+/-7 days).

In [1]:
from pathlib import Path
import sys, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
sys.path.insert(0, "../../src")

from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

import models.forecasting_dl as fdl
import models.recursive_rollout as rr
from evaluation.metrics import full_metrics

HOURLY = Path("../../data/Hourly"); RESULTS = Path("../../results")
ANCHOR = pd.Timestamp("2021-12-16")
N_DAYS = 365
TOWER = 4
DUM = ["is_t2", "is_t4", "is_t9"]
AR_COLS = ["ar_ch4_dlag1", "ar_ch4_dlag2", "ar_ch4_dlag3", "ar_ch4_dlag7", "ar_ch4_dlag14", "ar_ch4_drm7"]
EXOG_B = ["fx_lsu_dens", "fx_WS_mean", "fx_VPD_mean", "fx_USTAR_mean", "fx_PPFD_mean",
          "fx_DOY_sin", "fx_DOY_cos", "fx_is_growing"]
print(f"Anchor: {ANCHOR.date()}  ->  target window {(ANCHOR+pd.Timedelta(days=1)).date()} .. {(ANCHOR+pd.Timedelta(days=N_DAYS)).date()}")

Anchor: 2021-12-16  ->  target window 2021-12-17 .. 2022-12-16


## 1  Load data, confirm anchor/target-window coverage

In [2]:
dv = pd.read_csv(HOURLY/"forecast_daily_v2.csv", low_memory=False)
dv["Datetime"] = pd.to_datetime(dv["Datetime"], format="mixed")
FX_B = [c for c in dv.columns if c.startswith("fx")]
T = {t: dv[dv.tower == t].set_index("Datetime").sort_index() for t in [2, 4, 9]}

target_dates = pd.date_range(ANCHOR + pd.Timedelta(days=1), periods=N_DAYS, freq="D")
n_real = T[TOWER].loc[target_dates, "y_observed"].notna().sum()
print(f"Tower {TOWER}: {n_real}/{N_DAYS} days in target window pass the y_observed>=6h/day rule "
      f"({100*n_real/N_DAYS:.1f}%) -- true hourly-observed density is lower still (~48.9%, see D-51-style caveat).")
print(f"Pre-anchor real history: {len(T[TOWER].loc[:ANCHOR])} rows")

Tower 4: 341/365 days in target window pass the y_observed>=6h/day rule (93.4%) -- true hourly-observed density is lower still (~48.9%, see D-51-style caveat).
Pre-anchor real history: 1811 rows


## 2  Tree models (RandomForest, XGBoost, LightGBM)

Iterative wrapper reusing `build_forecasting_matrix_v2.py`'s exact `.shift()`/`.rolling()` AR-feature
math (`recursive_rollout.ar_features_for_day`) -- only the 6 `ar_ch4_*` columns are recursion-
dependent; every `fx_*`/`ar_fc_dlag1` feature is real historical data (perfect foresight). Pooled
training (Towers 2/4/9, tower dummies, D-30 default) truncated to <= anchor. Hyperparameters reused
verbatim from B-03's daily-track Round-1 HPO (RF `min_samples_leaf=10,max_features=0.5`; XGB
`max_depth=2,learning_rate=0.02,n_estimators=400,min_child_weight=10`) -- no new HPO (D-41).
LightGBM (net-new to this project) mirrors XGB's config directly.

In [3]:
def fit_tree(algo, tr, feat_cols):
    imp = SimpleImputer(strategy="mean"); Xi = imp.fit_transform(tr[feat_cols].values)
    if algo == "RF":
        m = RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=42,
                                   min_samples_leaf=10, max_features=0.5)
    elif algo == "XGB":
        m = XGBRegressor(subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42,
                          max_depth=2, learning_rate=0.02, n_estimators=400, min_child_weight=10)
    elif algo == "LightGBM":
        m = LGBMRegressor(subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42,
                           num_leaves=7, min_child_samples=10, learning_rate=0.02, n_estimators=400,
                           verbosity=-1)
    else:
        raise ValueError(algo)
    m.fit(Xi, tr["target"].values); return m, imp

feat_cols = AR_COLS + FX_B + ["ar_fc_dlag1"] + DUM
pool = []
for t in [2, 4, 9]:
    df = T[t].copy(); df["target"] = df["y_gapfilled"]
    for d in DUM: df[d] = 1.0 if d == f"is_t{t}" else 0.0
    pool.append(df[df.index <= ANCHOR])
tr = pd.concat(pool); tr = tr[tr["target"].notna()]
print(f"Pooled training rows (<= anchor): {len(tr)}")

df4 = T[TOWER]
history_init = df4.loc[:ANCHOR, "y_gapfilled"].copy()
fx_frame = df4.loc[target_dates, FX_B + ["ar_fc_dlag1"]].copy()
fx_frame["is_t2"], fx_frame["is_t4"], fx_frame["is_t9"] = 0.0, 1.0, 0.0

tree_chains = {}
for algo in ["RF", "XGB", "LightGBM"]:
    t0 = time.time()
    model, imp = fit_tree(algo, tr, feat_cols)
    tree_chains[algo] = rr.tree_rollout(model, imp, feat_cols, fx_frame, history_init, ANCHOR, n_days=N_DAYS)
    print(f"  {algo} rollout done ({time.time()-t0:.0f}s)", flush=True)

Pooled training rows (<= anchor): 5433


  RF rollout done (18s)


  XGB rollout done (1s)


  LightGBM rollout done (1s)


## 3  SARIMAX

No day-by-day refeed loop needed -- this is SARIMAX's one genuinely "free" case. Order search
(`search_order`, bounded AIC grid `d=1,p in {1,2},q in {0,1}`) reused verbatim from `B03a_arima.ipynb`,
refit on data <= anchor. **Not** `fit_walk_forward()` -- that periodically re-observes real ground
truth (`append(refit=False)`), exactly the "cheating" mechanism this experiment tests against.
Instead: fit once through the anchor, then call `get_forecast(steps=365)` **exactly once**, with
real historical `EXOG_B` values for the target window as perfect-foresight exogenous input.

In [4]:
y = df4["y_gapfilled"].astype(float)
X = df4[EXOG_B].astype(float).ffill().bfill()
y_tr, X_tr = y.loc[:ANCHOR], X.loc[:ANCHOR]

best = None
for p in [1, 2]:
    for q in [0, 1]:
        try:
            t0 = time.time()
            m = SARIMAX(y_tr, exog=X_tr, order=(p, 1, q), enforce_stationarity=False, enforce_invertibility=False)
            res = m.fit(disp=False, maxiter=50)
            print(f"    order=({p},1,{q}) aic={res.aic:.1f} ({time.time()-t0:.1f}s)", flush=True)
            if best is None or res.aic < best[0]: best = (res.aic, (p, 1, q), res)
        except Exception as e:
            print(f"    order=({p},1,{q}) failed: {e}", flush=True)
sarimax_order, sarimax_res = best[1], best[2]
future_X = X.loc[target_dates]
fc = sarimax_res.get_forecast(steps=N_DAYS, exog=future_X)
sarimax_chain = pd.Series(fc.predicted_mean.values, index=target_dates)
print(f"SARIMAX order={sarimax_order}, single get_forecast(steps={N_DAYS}) done")

C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


    order=(1,1,0) aic=17632.9 (0.8s)


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


    order=(1,1,1) aic=17337.1 (1.2s)


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


    order=(2,1,0) aic=17555.8 (1.2s)


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)


    order=(2,1,1) aic=17310.3 (1.5s)


SARIMAX order=(2, 1, 1), single get_forecast(steps=365) done


C:\ProgramData\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


## 4  DL models (DLinear, LSTM)

A genuinely new slide-forward harness (`recursive_rollout.dl_rollout`) -- no recursive/refeed
mechanism exists anywhere in `forecasting_dl.py`'s batch-oriented `run_track()`. Freezes the
existing trained `L=28,H=14` architecture; at each day only `pred[:,0]` (the first predicted day)
is kept and fed back into the growing CH4 history, sliding the 28-day encoder window forward one
day at a time. Note the daily DL track uses a different feature source than the tree track
(`forecast_features_v2.csv`, hourly resampled to daily, vs. `forecast_daily_v2.csv` directly) --
an existing B-03-vs-B-04 asymmetry, not new to this experiment.

In [5]:
device = fdl.get_device(); print("device:", device)
track = "B"; cfg = fdl.TRACKS[track]
m = fdl.load_matrix(HOURLY/"forecast_features_v2.csv")
W = fdl.build_windows(m, track)
cutoff = pd.Timestamp("2021-12-16 23:59")
tr_parts = [fdl._subset(W[t], pd.DatetimeIndex(W[t]["ttime"][:, -1]) <= cutoff) for t in [2, 4, 9]]
train = fdl._cat(tr_parts)
se, sd = fdl.Scaler().fit(train["enc"]), fdl.Scaler().fit(train["dec"])
yv = train["y"][np.isfinite(train["y"])]; mu, sdy = float(yv.mean()), float(yv.std()+1e-6)
train["enc"], train["dec"] = se.tf(train["enc"]), sd.tf(train["dec"])
n_enc, n_dec = train["enc"].shape[-1], train["dec"].shape[-1]

ser4 = fdl.tower_series(m, TOWER, track)
dates_full, enc_ex_full, dec_ex_full = ser4["idx"], ser4["enc_ex"], ser4["dec_ex"]
Y_full_real, ch4_full_real = ser4["Y"], ser4["ch4"]
anchor_idx = dates_full.get_loc(ANCHOR)
dl_history_init = ch4_full_real[:anchor_idx+1]

dl_chains = {}
for name in ["DLinear", "LSTM"]:
    t0 = time.time()
    model = fdl.build_model(name, cfg["L"], cfg["H"], n_enc, n_dec, 3)
    fdl.train_model(model, train, device, epochs=30, ch4_mu=mu, ch4_sd=sdy, seed=0)
    dl_chains[name] = rr.dl_rollout(model, se, sd, mu, sdy, device, fdl.TOW[TOWER],
                                     enc_ex_full, dec_ex_full, dates_full, dl_history_init, ANCHOR,
                                     L=cfg["L"], H=cfg["H"], n_days=N_DAYS)
    print(f"  {name} rollout done ({time.time()-t0:.0f}s)", flush=True)

device: cuda


  DLinear rollout done (2s)


  LSTM rollout done (2s)


## 5  Baselines, ground truth, binned evaluation

In [6]:
y_true_full = pd.Series(Y_full_real, index=dates_full)
y_true = y_true_full.reindex(target_dates).values
anchor_val = df4.loc[ANCHOR, "y_gapfilled"]
persist = rr.chain_persistence(anchor_val, N_DAYS)
hist_obs = df4.loc[:ANCHOR - pd.Timedelta(days=1), "y_observed"].dropna()
clim = rr.doy_climatology(hist_obs, target_dates, window=7)

all_chains = {**tree_chains, "SARIMAX": sarimax_chain, **dl_chains,
              "Persistence": pd.Series(persist, index=target_dates),
              "Climatology": pd.Series(clim, index=target_dates)}

rows = []
for name, chain in all_chains.items():
    yp = chain.reindex(target_dates).values
    bm = rr.bin_metrics(y_true, yp, target_dates, ANCHOR, y_persist=persist)
    bm["model"] = name
    rows.append(bm)
R = pd.DataFrame(pd.concat(rows, ignore_index=True))
R.to_csv(RESULTS/"b09_summary.csv", index=False)

chains_df = pd.DataFrame({name: c.reindex(target_dates) for name, c in all_chains.items()})
chains_df["y_true"] = y_true
chains_df.to_csv(RESULTS/"b09_chains.csv")

pd.set_option("display.width", 200)
print("=== B-09 binned R2 by model (n per bin shown separately below -- 1-7 has only n=3, interpret with caution) ===")
print(R.pivot_table(index="model", columns="bin", values="R2").round(3).to_string())
print("\n=== B-09 binned MASE by model (<1 beats chain-persistence) ===")
print(R.pivot_table(index="model", columns="bin", values="MASE").round(3).to_string())
print("\n=== n real y_observed days per bin ===")
print(R[R.model=="RF"][["bin","n"]].to_string(index=False))
print(f"\nSARIMAX order: {sarimax_order}")

=== B-09 binned R2 by model (n per bin shown separately below -- 1-7 has only n=3, interpret with caution) ===
bin             1-7  181-270  271-365  31-90   8-30  91-180
model                                                      
Climatology   0.020   -0.192   -0.274 -0.223 -0.331  -0.141
DLinear     -31.181    0.417   -1.017 -2.628 -5.555   0.260
LSTM        -15.351    0.181   -0.642 -0.141 -0.658  -0.123
LightGBM     -0.606    0.335   -0.582 -0.056 -0.214   0.193
Persistence  -0.313   -0.254   -0.009 -0.077  0.000  -0.383
RF           -3.127    0.222   -0.594 -0.182 -0.732   0.209
SARIMAX      -5.577    0.207   -0.271  0.016  0.013   0.111
XGB          -1.803    0.322   -0.514 -0.098 -0.170   0.137

=== B-09 binned MASE by model (<1 beats chain-persistence) ===
bin            1-7  181-270  271-365  31-90   8-30  91-180
model                                                     
Climatology  0.906    1.157    1.199  1.286  1.138   0.812
DLinear      4.808    0.872    1.736  2.338  2.8

## 6  Figure -- full 365-day chain vs. real observations

One time-series figure showing every model's chain against real `y_observed` and both baselines --
visually shows where/if divergence starts, alongside the binned-metric table above.

In [7]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 6))
ax.scatter(chains_df.index, chains_df["y_true"], s=10, color="black", label="real y_observed", zorder=5)
for name in ["RF", "XGB", "LightGBM", "SARIMAX", "DLinear", "LSTM"]:
    ax.plot(chains_df.index, chains_df[name], lw=1, alpha=0.7, label=name)
ax.plot(chains_df.index, chains_df["Persistence"], lw=1.5, ls="--", color="gray", label="Persistence")
ax.plot(chains_df.index, chains_df["Climatology"], lw=1.5, ls=":", color="dimgray", label="Climatology")
ax.set_title(f"B-09: Tower {TOWER} recursive 365-day rollout, anchor {ANCHOR.date()}")
ax.set_ylabel("FCH4 (nmol m-2 s-1)"); ax.legend(ncol=4, fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(RESULTS/"figures/b09_rollout_chains.png", dpi=120)
print("saved results/figures/b09_rollout_chains.png")

saved results/figures/b09_rollout_chains.png


## 7  Append to benchmarks.csv (B09)

In [8]:
bench = RESULTS/"benchmarks.csv"; today = pd.Timestamp.today().date().isoformat()
ex = pd.read_csv(bench); ex = ex[ex["replication"] != "B09"]
rows = []
for _, r in R.iterrows():
    lo, hi = r["bin"].split("-")
    rows.append({"replication": "B09", "model": r["model"], "tower": f"Tower {TOWER}",
        "feature_set": "recursive 365-day single-anchor rollout backtest (anchor 2021-12-16); perfect-foresight fx_ drivers",
        "track": "B", "horizon": int(hi), "split": f"b09_leadtime_bin_{r['bin']}",
        "R2": r["R2"], "MAE": r["MAE"], "n_test": int(r["n"]),
        "MASE": r["MASE"],
        "date": today,
        "notes": f"B09 recursive rollout backtest, lead-time bin days {r['bin']}; SARIMAX order={sarimax_order} where applicable; D-46 requirement-5 precursor test"})
new = pd.DataFrame(rows); comb = pd.concat([ex, new], ignore_index=True); comb.to_csv(bench, index=False)
print(f"Wrote {len(new)} B09 rows. Total {len(comb)}.")

Wrote 48 B09 rows. Total 3785.


## 8  Stretch items (not attempted this session -- see `b09_results.md` for status)

- **TFT**: known instability (D-45/D-48) makes it hard to distinguish genuine recursive
  compounding from pre-existing shakiness -- deferred.
- **Tower 9**: confirmatory, not new-mechanism information -- deferred.
- **TabPFN / tabpfn-time-series**: not installed (`pip install tabpfn tabpfn-time-series`);
  context-length limits for 365 autoregressive steps are unresearched -- prep-only, no run
  attempted.